# The Evolution of Space Exploration (1957–2023)

## Project Overview

This project explores the history of space exploration using a dataset of more than 6,700 space missions.

The analysis focuses on four key questions:

1. How has space exploration developed since 1957?
2. How has the balance of power in space exploration changed over time?
3. How reliable has space exploration become?
4. Has private spaceflight created a new era of space exploration?

## Tools Used

- Python
- Pandas
- Matplotlib
- Jupyter Notebook

## Dataset

The dataset contains historical space mission records from 1957 to 2023, including mission outcomes, launch locations, organizations, and launch dates.

# 1. Import Libraries

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


# 2. Load Dataset

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "all_space_mission_launches.csv"
IMAGES_DIR = PROJECT_ROOT / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

df.head()


# 3. Data Understanding

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

# 4. Data Cleaning

## Cleaning Steps

The following transformations were performed:

- Removed unnecessary columns
- Extracted country information from launch locations
- Standardized date format
- Reordered columns
- Removed incomplete data for 2024

In [ ]:
df = df.drop(columns=["Unnamed: 0", "Price"])

df.head()

In [ ]:
df["Country"] = df["Location"].str.split(",").str[-1].str.strip()

df.head()

In [ ]:
df["Location"] = df["Location"].str.rsplit(",", n=1).str[0]

df.head()

In [ ]:
df["Date"] = df["Datetime"].str.extract(r'([A-Za-z]{3}\s[A-Za-z]{3}\s\d{2},\s\d{4})')

df["Date"] = pd.to_datetime(
    df["Date"],
    format="%a %b %d, %Y"
)

df["Date"] = df["Date"].dt.strftime("%d/%m/%Y")

df = df.drop(columns=["Datetime"])

df.head()

In [ ]:
df = df[
    [
        "Organisation",
        "Location",
        "Country",
        "Details",
        "Date",
        "Rocket_Status",
        "Mission_Status"
    ]
]

df.head()

# 5. Exploratory Data Analysis

In [ ]:
df["Organisation"].nunique()

In [ ]:
df["Organisation"].value_counts().head(15)

In [ ]:
df["Country"].nunique()

In [ ]:
df["Country"].value_counts().head(15)

In [ ]:
df["Mission_Status"].value_counts()

In [ ]:
date_series = pd.to_datetime(df["Date"], format="%d/%m/%Y")

print("First Year:", date_series.dt.year.min())
print("Last Year:", date_series.dt.year.max())

In [ ]:
df["Year"] = pd.to_datetime(
    df["Date"],
    format="%d/%m/%Y"
).dt.year

# Remove incomplete year
df = df[df["Year"] < 2024]

In [ ]:
missions_per_year = (
    df.groupby("Year")
      .size()
      .reset_index(name="Mission_Count")
)

missions_per_year

In [ ]:
plt.figure(figsize=(14, 7))

plt.plot(
    missions_per_year["Year"],
    missions_per_year["Mission_Count"],
    linewidth=2
)

# Important historical events
events = {
    1957: "Sputnik 1",
    1969: "Moon landing",
    1991: "End of USSR",
    2002: "SpaceX founded",
    2020: "Modern launch boom"
}

for year, label in events.items():
    plt.axvline(x=year, linestyle="--", alpha=0.6)
    plt.text(
        year + 0.3,
        missions_per_year["Mission_Count"].max() * 0.9,
        label,
        rotation=90,
        verticalalignment="top"
    )

plt.title("Evolution of Space Missions (1957–2023)", fontsize=16)
plt.xlabel("Year")
plt.ylabel("Number of Missions")
plt.grid(alpha=0.3)

plt.savefig(
    IMAGES_DIR / "evolution_of_space_missions.png",
    bbox_inches="tight",
    dpi=300
)

plt.show()

## Question 1: How Has Space Exploration Developed Since 1957?

The chart shows the number of space missions per year from 1957 to 2023.  
The year 2024 was excluded because the dataset only contains data for part of that year.

### Key Findings

#### 1. Rapid Growth During the Space Race

Space activity increased strongly after the launch of Sputnik 1 in 1957.  
During the 1960s, the number of annual missions rose quickly, reflecting the intense competition between the United States and the Soviet Union.

#### 2. High Activity During the Cold War

From the late 1960s through the 1980s, space missions remained at a consistently high level.  
This period was shaped by military, scientific, and political competition.

#### 3. Decline After the End of the Soviet Union

After 1991, the number of annual missions declined noticeably.  
This suggests that the end of the Cold War and the collapse of the Soviet Union had a major impact on global launch activity.

#### 4. A New Space Boom Since the 2010s

From around 2010 onward, the number of missions started to rise again.  
By 2023, the dataset reaches its highest annual number of missions, indicating a new era of space activity.

### Conclusion

The development of space missions follows a clear historical pattern:  
an early Space Race boom, a long Cold War period of high activity, a post-Soviet decline, and a modern resurgence driven by new actors and commercial spaceflight.

# Question 2: How Did Launch Activity Shift Among Selected Organization Groups?

The following comparison uses a manually defined mapping from launch organizations to broad national or regional groups. Only organizations included in the mapping are counted, so this section should be interpreted as a portfolio-style comparison of major actors rather than a complete geopolitical accounting of all missions.


In [ ]:
nation_map = {
    "RVSN USSR": "USSR",
    "Roscosmos": "Russia",
    "VKS RF": "Russia",
    "ILS": "Russia",
    "Kosmotras": "Russia",
    "Eurockot": "Russia",

    "NASA": "USA",
    "US Air Force": "USA",
    "General Dynamics": "USA",
    "ULA": "USA",
    "Boeing": "USA",
    "Martin Marietta": "USA",
    "Northrop": "USA",
    "Lockheed": "USA",
    "SpaceX": "USA",
    "Blue Origin": "USA",
    "US Navy": "USA",
    "Sea Launch": "USA",
    "Virgin Galactic": "USA",
    "ABMA": "USA",
    "Astra": "USA",
    "Virgin Orbit": "USA",
    "Firefly": "USA",
    "Relativity Space": "USA",
    "ABL SS": "USA",

    "CASC": "China",
    "ExPace": "China",
    "Galactic Energy": "China",
    "CASIC": "China",
    "Landspace": "China",
    "i-Space": "China",
    "OneSpace": "China",

    "ISRO": "India",

    "MHI": "Japan",
    "ISAS": "Japan",
    "JAXA": "Japan",

    "Arianespace": "Europe",
    "ESA": "Europe",
    "Starsem": "Europe",
    "CNES": "Europe",
    "ASI": "Europe",
    "Armée de l'Air": "Europe",

    "Rocket Lab": "New Zealand",

    "IAI": "Israel",

    "ISA": "Iran",
    "IRGC": "Iran",

    "KARI": "South Korea",
}

In [ ]:
df["Nation"] = df["Organisation"].map(nation_map)

nation_coverage = df["Nation"].notna().mean()
print(f"Share of missions covered by nation mapping: {nation_coverage:.1%}")


In [ ]:
missions_by_nation_year = (
    df.dropna(subset=["Nation"])
      .groupby(["Year", "Nation"])
      .size()
      .reset_index(name="Mission_Count")
)

missions_by_nation_year


In [ ]:
top_nations = (
    df["Nation"]
    .value_counts()
    .head(5)
    .index
)

top_nations

In [ ]:
top_nation_data = missions_by_nation_year[
    missions_by_nation_year["Nation"].isin(top_nations)
]

In [ ]:
plt.figure(figsize=(14, 7))

for nation in top_nations:
    nation_data = top_nation_data[
        top_nation_data["Nation"] == nation
    ]

    plt.plot(
        nation_data["Year"],
        nation_data["Mission_Count"],
        label=nation,
        linewidth=2
    )

plt.title(
    "Space Missions by Nation Over Time",
    fontsize=16
)

plt.xlabel("Year")
plt.ylabel("Number of Missions")
plt.legend()

plt.grid(alpha=0.3)

plt.savefig(
    IMAGES_DIR / "space_power_balance.png",
    bbox_inches="tight",
    dpi=300
)

plt.show()

## Interpretation

The mapped organizations show major shifts in launch activity over time, including strong early Soviet activity, sustained U.S.-linked activity, and substantial growth among Chinese organizations in recent decades.

Because the mapping is manually curated and incomplete, the chart is descriptive of the classified subset and should not be treated as an authoritative ranking of national space programs.


In [ ]:
df["Mission_Status"].value_counts()

In [ ]:
df["Successful"] = (
    df["Mission_Status"] == "Success"
)

In [ ]:
success_rate = (
    df.groupby("Year")["Successful"]
      .mean()
      .reset_index(name="Success_Rate")
)

success_rate["Success_Rate"] = success_rate["Success_Rate"] * 100

success_rate.head(20)

In [ ]:
plt.figure(figsize=(14, 7))

plt.plot(
    success_rate["Year"],
    success_rate["Success_Rate"],
    linewidth=2
)

plt.title("Success Rate of Space Missions Over Time", fontsize=16)
plt.xlabel("Year")
plt.ylabel("Success Rate (%)")

plt.ylim(0, 100)
plt.grid(alpha=0.3)

plt.savefig(
    IMAGES_DIR / "mission_success_rate_by_year.png",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
df["Decade"] = (df["Year"] // 10) * 10

success_by_decade = (
    df.groupby("Decade")["Successful"]
      .mean()
      .reset_index(name="Success_Rate")
)

success_by_decade["Success_Rate"] *= 100

success_by_decade

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.bar(
    success_by_decade["Decade"].astype(str),
    success_by_decade["Success_Rate"]
)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 1,
        f"{height:.1f}%",
        ha="center"
    )

plt.title("Average Mission Success Rate by Decade")
plt.xlabel("Decade")
plt.ylabel("Success Rate (%)")

plt.ylim(0, 105)

plt.savefig(
    IMAGES_DIR / "mission_success_rate_by_decade.png",
    bbox_inches="tight",
    dpi=300
)

plt.show()

# Question 3: How Reliable Has Space Exploration Become?

This analysis examines how mission success rates have changed over time.

A mission was considered successful if its status was recorded as "Success". All other outcomes, including failures and partial failures, were treated as unsuccessful missions.

## Key Findings

### 1. Early Space Exploration Was Highly Risky

During the 1950s, only about one third of missions were successful.

This reflects the experimental nature of early rocket technology and the challenges faced by the first space programs.

### 2. Reliability Improved Rapidly During the 1960s

The success rate increased dramatically from approximately 32% in the 1950s to more than 80% in the 1960s.

This was the largest improvement observed in the entire dataset.

### 3. Modern Spaceflight Reached High Reliability Early

By the 1970s, mission success rates had already exceeded 90%.

Within this dataset, recorded mission success rates exceeded 90% by the 1970s. This descriptive result does not by itself identify the engineering or operational causes of the improvement.

### 4. Reliability Has Remained Stable for More Than Fifty Years

Since the 1970s, success rates have consistently remained between roughly 92% and 95%.

Recorded success rates remain high in later decades, although mission types and reporting practices vary across time.

## Conclusion

The most important finding is not that space missions are reliable today, but that they became reliable surprisingly early.

Most of the progress occurred between the 1950s and 1970s, after which mission success rates remained consistently high for decades.

In [ ]:
sector_map = {

    # Public
    "NASA": "Public",
    "US Air Force": "Public",
    "US Navy": "Public",
    "RVSN USSR": "Public",
    "Roscosmos": "Public",
    "VKS RF": "Public",
    "CASC": "Public",
    "ISRO": "Public",
    "ESA": "Public",
    "JAXA": "Public",
    "CNES": "Public",
    "ISAS": "Public",
    "ISA": "Public",
    "IRGC": "Public",
    "KARI": "Public",

    # Private
    "SpaceX": "Private",
    "Rocket Lab": "Private",
    "Blue Origin": "Private",
    "Virgin Galactic": "Private",
    "Virgin Orbit": "Private",
    "ULA": "Private",
    "Boeing": "Private",
    "Lockheed": "Private",
    "Northrop": "Private",
    "General Dynamics": "Private",
    "Martin Marietta": "Private",
    "Arianespace": "Private",
    "Sea Launch": "Private",
    "Starsem": "Private",
    "ILS": "Private",
    "Kosmotras": "Private",
    "ExPace": "Private",
    "Galactic Energy": "Private",
    "Landspace": "Private",
    "i-Space": "Private",
    "OneSpace": "Private",
    "Firefly": "Private",
    "Relativity Space": "Private",
    "ABL SS": "Private",
    "Astra": "Private"
}

In [ ]:
df["Sector"] = df["Organisation"].map(sector_map)

sector_coverage = df["Sector"].notna().mean()
print(f"Share of missions covered by public/private mapping: {sector_coverage:.1%}")


In [ ]:
missions_by_sector = (
    df.dropna(subset=["Sector"])
      .groupby(["Year", "Sector"])
      .size()
      .reset_index(name="Mission_Count")
)


In [ ]:
plt.figure(figsize=(14,7))

for sector in missions_by_sector["Sector"].dropna().unique():

    sector_data = missions_by_sector[
        missions_by_sector["Sector"] == sector
    ]

    plt.plot(
        sector_data["Year"],
        sector_data["Mission_Count"],
        label=sector,
        linewidth=2
    )

plt.title(
    "Public vs Private Space Missions Over Time",
    fontsize=16
)

plt.xlabel("Year")
plt.ylabel("Number of Missions")

plt.legend()

plt.grid(alpha=0.3)

plt.savefig(
    IMAGES_DIR / "public_vs_private_spaceflight.png",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
def classify_org(org):
    if org == "SpaceX":
        return "SpaceX"
    sector = sector_map.get(org)
    if sector == "Private":
        return "Other Private Companies"
    if sector == "Public":
        return "Public Organizations"
    return pd.NA


df["Org_Group"] = df["Organisation"].apply(classify_org)


In [ ]:
comparison = (
    df.dropna(subset=["Org_Group"])
      .groupby(["Year", "Org_Group"])
      .size()
      .reset_index(name="Mission_Count")
)


In [ ]:
plt.figure(figsize=(14, 7))

for group in [
    "Public Organizations",
    "Other Private Companies",
    "SpaceX"
]:

    group_data = comparison[
        comparison["Org_Group"] == group
    ]

    plt.plot(
        group_data["Year"],
        group_data["Mission_Count"],
        linewidth=2,
        label=group
    )

plt.title(
    "SpaceX vs Other Private Companies vs Public Organizations",
    fontsize=16
)

plt.xlabel("Year")
plt.ylabel("Number of Missions")

plt.legend()

plt.grid(alpha=0.3)

plt.savefig(
    IMAGES_DIR / "spacex_effect.png",
    bbox_inches="tight",
    dpi=300
)

plt.show()

# Question 4: How Did Launch Activity Shift Between Classified Public and Private Organizations?

This section uses a manually defined, time-invariant classification of selected launch organizations as public or private. Organizations not included in the mapping are excluded from this comparison.

## Key Findings

* In the classified subset, public organizations dominate much of the early period.
* Private launch organizations become increasingly prominent in more recent years.
* The composition changes strongly after the mid-2010s, with SpaceX accounting for a large share of classified private launch activity.

## Important Limitation

Public/private status can be historically complex, and some organizations have mixed ownership or changed structure over time. The mapping is therefore a simplified analytical classification rather than a definitive institutional taxonomy.


# Bonus Analysis: The SpaceX Effect

To examine the recent rise of commercial launch activity, classified organizations are divided into three groups:

- SpaceX
- other organizations classified as private
- organizations classified as public

Unmapped organizations are excluded.

## Key Finding

Within this classified subset, SpaceX accounts for a very large share of the recent increase in private launch activity. This supports the narrower conclusion that SpaceX is an important contributor to the modern launch boom represented in this dataset.


# Final Conclusions

The dataset shows a long-run increase in launch activity with distinct historical phases, improving recorded mission success rates, and a strong rise in recent launch activity.

The organization-level comparisons also show that the composition of launch activity has changed substantially over time. Because the national and public/private classifications rely on manually curated mappings, those sections are best interpreted as exploratory comparisons of major classified organizations rather than exhaustive geopolitical or institutional rankings.

Overall, the project demonstrates data cleaning, feature engineering, time-series aggregation, categorical mapping, visualization, and careful interpretation of historical launch data.
